In [2]:
import torch
from src.PhaseRetrievalProblem import PhaseRetrievalProblem
from src.ModelBasedSolver import ModelBasedSolver
from src.WeaklyConvexProblem import WeaklyConvexProblem
from src.problems.max_parabola import max_parabola_phi, max_parabola_model_gen
from src.regularizers import elastic_net_regularizer

In [3]:
# Setup Synthetic Data for Phase Retrieval
torch.manual_seed(42)
d = 10
m = 100
true_x = torch.randn(d)
a_matrix = torch.randn(m, d)
b_vector = (a_matrix @ true_x)**2 

# Combine into a population: each row is [a_1, ..., a_d, b]
population_data = torch.cat([a_matrix, b_vector.unsqueeze(1)], dim=1)

# Initialize Problem and Solver
prob = PhaseRetrievalProblem(rho=2.0)
solver = ModelBasedSolver(
    problem=prob, 
    data=population_data, 
    x_init=torch.randn(d), 
    T=500, 
    gamma=0.1
)

final_x = solver.run()
print(f"Final sampled x: {torch.norm(final_x).item():.4f}")
print(f"Final reconstruction error: {torch.norm(final_x - true_x).item():.4f}")

Iter    0: x = 3.4600 | True phi(x) = 9.0899
Iter   50: x = 3.0227 | True phi(x) = 4.0209
Iter  100: x = 2.7715 | True phi(x) = 1.6202
Iter  150: x = 2.6419 | True phi(x) = 0.4002
Iter  200: x = 2.6672 | True phi(x) = 0.0422
Iter  250: x = 2.6660 | True phi(x) = 0.0110
Iter  300: x = 2.6648 | True phi(x) = 0.0003
Iter  350: x = 2.6648 | True phi(x) = 0.0000
Iter  400: x = 2.6648 | True phi(x) = 0.0000
Iter  450: x = 2.6648 | True phi(x) = 0.0000
Final sampled x: 2.6648
Final reconstruction error: 0.0000


In [4]:
# Create a population of 1000 data points (targets for the parabolas)
# Center them around 0 and 4 so the "average" intersection is near 2
torch.manual_seed(42)
population_data = torch.randn(1000, 2)
population_data[:, 0] += 0.0  # Centers for f1
population_data[:, 1] += 4.0  # Centers for f2
population_data[:, 1] += 4.0  # Centers for f2
# 1. Setup Problem
# rho=2 because the components (x-xi)^2 have a second derivative of 2
prob = WeaklyConvexProblem(
    f_objective=max_parabola_phi,
    r_objective=elastic_net_regularizer,
    model_gen=max_parabola_model_gen,
    rho=2.0
)

# 2. Run with Mini-Batching
solver = ModelBasedSolver(
    problem=prob,
    data=population_data,
    x_init=torch.tensor([10.0]),
    T=1000,
    batch_size=32,
    gamma=0.5
)

final_x = solver.run()
print(f"Final sampled x: {final_x.item():.4f}")

Iter    0: x = 5.5768 | True phi(x) = 32.3955
Iter   50: x = 3.9014 | True phi(x) = 21.8445
Iter  100: x = 3.9698 | True phi(x) = 21.7957
Iter  150: x = 3.9300 | True phi(x) = 21.8188
Iter  200: x = 3.9974 | True phi(x) = 21.7908
Iter  250: x = 4.0221 | True phi(x) = 21.7947
Iter  300: x = 4.1257 | True phi(x) = 21.8835
Iter  350: x = 4.0212 | True phi(x) = 21.7945
Iter  400: x = 3.9385 | True phi(x) = 21.8126
Iter  450: x = 3.9587 | True phi(x) = 21.8005
Iter  500: x = 3.9598 | True phi(x) = 21.7999
Iter  550: x = 3.9833 | True phi(x) = 21.7922
Iter  600: x = 4.0877 | True phi(x) = 21.8356
Iter  650: x = 3.9803 | True phi(x) = 21.7928
Iter  700: x = 3.9969 | True phi(x) = 21.7908
Iter  750: x = 4.0624 | True phi(x) = 21.8138
Iter  800: x = 3.9681 | True phi(x) = 21.7963
Iter  850: x = 4.0355 | True phi(x) = 21.7989
Iter  900: x = 3.9991 | True phi(x) = 21.7909
Iter  950: x = 3.9165 | True phi(x) = 21.8300
Final sampled x: 4.0274
